In [41]:
import networkx as nx
import folium
from pyproj import Transformer
import math
from collections import defaultdict

In [43]:
# Load GraphML
G = nx.read_graphml("london_tubenetwork.graphml")
print("Graph loaded:", G.number_of_nodes(), "nodes,", G.number_of_edges(), "edges")

Graph loaded: 438 nodes, 486 edges


In [45]:
# Convert coords to WGS84 (lat/lon)
transformer = Transformer.from_crs("EPSG:27700", "EPSG:4326", always_xy=True)
valid_coords_count = 0
for n, data in G.nodes(data=True):
    coords = data.get("coords")
    if coords:
        x, y = map(float, coords.strip("()").split(","))
        data["x"] = x
        data["y"] = y
        lon, lat = transformer.transform(x, y)
        data["lat"] = lat
        data["lon"] = lon
        valid_coords_count += 1
    else:
        data["x"], data["y"] = None, None
        data["lat"], data["lon"] = None, None
print(valid_coords_count, "nodes have valid coordinates")

438 nodes have valid coordinates


In [61]:
# Compute betweenness centrality
betweenness = nx.betweenness_centrality(G, normalized=True)
nx.set_node_attributes(G, betweenness, "betweenness")

# Print top stations by betweenness centrality
print("\n" + "=" * 60)
print("TOP 10 STATIONS BY BETWEENNESS CENTRALITY")
print("=" * 60)
sorted_bc = sorted(betweenness.items(), key=lambda x: x[1], reverse=True)
for i, (node_id, bc_value) in enumerate(sorted_bc[:10], 1):
    station_name = G.nodes[node_id].get('station_name', node_id)
    print(f"{i:2d}. {station_name:30s} {bc_value:.6f}")


TOP 10 STATIONS BY BETWEENNESS CENTRALITY
 1. Baker Street                   0.381015
 2. Bethnal Green                  0.353433
 3. Finchley Road                  0.336582
 4. Bank                           0.319563
 5. Green Park                     0.319552
 6. Waterloo                       0.317216
 7. Liverpool Street               0.313026
 8. Westminster                    0.289962
 9. Bond Street                    0.258599
10. West Hampstead                 0.236566


In [49]:
# Define line colors
line_colors = {
    "Bakerloo": "#B36305",
    "Central": "#E32017",
    "Circle": "#FFD300",
    "District": "#00782A",
    "Hammersmith & City": "#F3A9BB",
    "Jubilee": "#A0A5A9",
    "Metropolitan": "#9B0056",
    "Northern": "#000000",
    "Piccadilly": "#0019A8",
    "Victoria": "#00A0E2",
    "Waterloo & City": "#95CDBA",
    "Elizabeth line": "#6950A1",
    "DLR": "#00A4A7",
    "London Overground": "#EE7C0E",
    "National Rail": "#808080"
}

In [51]:
# Build line-specific subgraphs
# Create a subgraph for each line
line_edges = defaultdict(list)

for u, v, edge_data in G.edges(data=True):
    u_data = G.nodes[u]
    v_data = G.nodes[v]
    
    if not (u_data.get("x") and v_data.get("x")):
        continue
    
    # Get lines from nodes
    u_lines = set(u_data.get('lines', '').split(',')) if u_data.get('lines') else set()
    v_lines = set(v_data.get('lines', '').split(',')) if v_data.get('lines') else set()
    shared_lines = u_lines & v_lines
    
    # Also use edge name
    if edge_data.get('name'):
        shared_lines.add(edge_data['name'])
    
    # Clean and add to each line
    for line in shared_lines:
        line = line.strip()
        if line:
            line_edges[line].append((u, v))

print(f"\nFound {len(line_edges)} different lines")


Found 18 different lines


In [53]:
# Find which edges have multiple lines (for offset calculation)
edge_to_lines = defaultdict(set)
for line, edges in line_edges.items():
    for u, v in edges:
        edge_key = tuple(sorted([u, v]))
        edge_to_lines[edge_key].add(line)

# Assign offset index to each line on each edge
edge_line_offsets = {}
for edge_key, lines in edge_to_lines.items():
    sorted_lines = sorted(lines)  # Consistent ordering
    for idx, line in enumerate(sorted_lines):
        edge_line_offsets[(edge_key, line)] = (idx, len(sorted_lines))

In [55]:
# Helper function for perpendicular offset
def get_offset_coords(x1, y1, x2, y2, offset_meters):
    """Offset a line perpendicular to its direction"""
    dx = x2 - x1
    dy = y2 - y1
    length = math.sqrt(dx**2 + dy**2)
    
    if length == 0:
        return x1, y1, x2, y2
    
    perp_x = dy / length
    perp_y = -dx / length
    
    return (x1 + perp_x * offset_meters, y1 + perp_y * offset_meters,
            x2 + perp_x * offset_meters, y2 + perp_y * offset_meters)

In [57]:
# Create Folium map
m = folium.Map(location=[51.5074, -0.1278], zoom_start=11, tiles="CartoDB positron")

# Draw each line as a continuous path
for line_name, edges in line_edges.items():
    color = line_colors.get(line_name, "gray")
    
    # Build line segments with consistent offsets
    line_segments = []
    
    for u, v in edges:
        u_data = G.nodes[u]
        v_data = G.nodes[v]
        
        x1, y1 = u_data["x"], u_data["y"]
        x2, y2 = v_data["x"], v_data["y"]
        
        edge_key = tuple(sorted([u, v]))
        idx, num_lines = edge_line_offsets.get((edge_key, line_name), (0, 1))
        
        # Calculate offset - REDUCED spacing
        if num_lines == 1:
            offset = 0
        elif num_lines == 2:
            offset = 3 if idx == 0 else -3  # Reduced from 30
        elif num_lines == 3:
            offsets = [-3, 0, 3]  # Reduced from 40
            offset = offsets[idx]
        else:
            spacing = 3  # Reduced from 30
            offset = (idx - (num_lines - 1) / 2) * spacing
        
        # Apply offset
        x1_off, y1_off, x2_off, y2_off = get_offset_coords(x1, y1, x2, y2, offset)
        
        # Convert to lat/lon
        lon1_off, lat1_off = transformer.transform(x1_off, y1_off)
        lon2_off, lat2_off = transformer.transform(x2_off, y2_off)
        
        line_segments.append([[lat1_off, lon1_off], [lat2_off, lon2_off]])
    
    # Draw all segments for this line
    for segment in line_segments:
        folium.PolyLine(
            locations=segment,
            color=color,
            weight=2.5,
            opacity=0.8,
            tooltip=line_name
        ).add_to(m)

# Draw nodes
max_bc = max(betweenness.values()) if betweenness else 1
for n, data in G.nodes(data=True):
    lat, lon = data.get("lat"), data.get("lon")
    if lat and lon:
        radius = 5 + 20 * (data.get("betweenness", 0) / max_bc)
        folium.CircleMarker(
            location=[lat, lon],
            radius=radius,
            popup=f"{data.get('station_name')}<br>Betweenness: {data.get('betweenness'):.4f}",
            color="red",
            fill=False,
            weight=2,
            opacity=0.9
        ).add_to(m)


# Save map
m.save("london_tube_offset_map.html")
print("\nMap saved to london_tube_offset_map.html")
m


Map saved to london_tube_offset_map.html
